# 07 – Monte Carlo, bizonytalanság és ritka események

**8. tanegység · javasolt idő: 2 × 50 perc**

## Tanulási célok

- Monte Carlo becslés és standard error értelmezése
- FTA eredmény szimulációs validálása
- paraméterbizonytalanság propagálása
- rare-event probléma és splitting/RESTART gondolat felismerése

**PUNDIT kapcsolat.** Ez a tanegység a PUNDIT projekt (`2020-1.2.3-EUREKA-2022-00021`) eredményeinek **oktatási hasznosítását** támogatja; önmagában nem helyettesíti és nem igazolja a projekt eredeti műszaki vállalásainak teljesítését.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Projekt gyökér:", ROOT)


## 1. Monte Carlo alapötlet

Sok véletlen realizációt generálunk, minden realizációban kiértékeljük a rendszert, majd a kimenetek gyakoriságából becslünk. Bernoulli eseménynél $\hat p=N_{event}/N$, és nagy mintánál $SE(\hat p)\approx\sqrt{\hat p(1-\hat p)/N}$.


In [ ]:
rng=np.random.default_rng(42); N=10_000; p=.02
x=rng.random(N)<p; phat=x.mean(); se=np.sqrt(phat*(1-phat)/N)
print(phat,se,x.sum())


## 2. Konvergencia

A mintaszám növelése nem monoton javulást jelent minden futásban, de a bizonytalanság nagyságrendje $1/\sqrt N$ szerint csökken.


In [ ]:
rows=[]
for n in [100,500,1_000,5_000,10_000,50_000,100_000]:
 r=np.random.default_rng(100+n); rows.append({'N':n,'estimate':(r.random(n)<p).mean()})
conv=pd.DataFrame(rows); plt.semilogx(conv.N,conv.estimate,'o-'); plt.axhline(p,ls='--'); plt.grid(alpha=.3); plt.show(); conv


## 3. TK-101 FTA Monte Carlo-val

A static FTA eseményeit független Bernoulli változókként generáljuk. Ez szándékosan ugyanazt a függetlenségi feltevést használja, mint az analitikus FTA, így validációs összehasonlításra alkalmas.


In [ ]:
prob={'CHALLENGE':.08,'LSHH_FOD':.01,'XV_FOD':.005,'LT_LOW':.02,'OP_MISS':.04}
def simulate(n=500_000,seed=42):
 r=np.random.default_rng(seed); s={k:r.random(n)<v for k,v in prob.items()}
 auto=s['LSHH_FOD']|s['XV_FOD']; manual=s['LT_LOW']|s['OP_MISS']; over=s['CHALLENGE']&auto&manual
 ph=over.mean(); return ph,np.sqrt(ph*(1-ph)/n),int(over.sum())
ph,se,events=simulate(); print(ph,se,events)
p_auto=1-(1-.01)*(1-.005); p_manual=1-(1-.02)*(1-.04); exact=.08*p_auto*p_manual
print('exact=',exact,'z=',(ph-exact)/se)


## 4. Paraméterbizonytalanság

Az input probability-k sem pontos számok. Oktatási példában Beta-eloszlást használunk. A Beta-paraméterek **szintetikusak**, nem üzemi bizonyítékok.


In [ ]:
r=np.random.default_rng(7); M=20_000
p_ch=r.beta(8,92,M); p_lshh=r.beta(2,198,M); p_xv=r.beta(1,199,M); p_lt=r.beta(4,196,M); p_op=r.beta(8,192,M)
p_auto=1-(1-p_lshh)*(1-p_xv); p_manual=1-(1-p_lt)*(1-p_op); p_over=p_ch*p_auto*p_manual
print(np.quantile(p_over,[.05,.5,.95])); plt.hist(p_over,bins=60); plt.xlabel('P(overflow)'); plt.show()


## 5. Ritka esemény probléma

Ha $p=10^{-7}$, akkor $N=10^6$ futásban a várható eseményszám csak 0.1. A nulla megfigyelés nem jelenti azt, hogy a valós probability nulla.


In [ ]:
rare_p=1e-7
for n in [10**4,10**5,10**6,10**7]: print(f'N={n:,}, várható eseményszám={n*rare_p:.3f}')


## 6. Splitting / RESTART gondolat

Ha a ritka végállapot felé vezető trajektórián vannak köztes ‘közeledési’ szintek, a szinteket elérő realizációkat újra felhasználhatjuk/sokszorozhatjuk. Így a számítási erőforrást a releváns trajektóriákra koncentráljuk. Az itt használt repository-függvény **pedagógiai multilevel splitting demo**, nem production-grade RESTART implementation.


In [ ]:
from safetycourse.simulation import random_walk_exceedance,multilevel_splitting_random_walk
crude,se=random_walk_exceedance(n=20_000,steps=50,drift=-.05,sigma=1.,threshold=20,seed=1)
split=multilevel_splitting_random_walk(particles=2_000,steps=50,drift=-.05,sigma=1.,levels=[8,14,20],seed=1)
print('crude',crude,'SE',se); print('splitting demo',split)


## Próbáld ki!

1. Csökkentsd az összes basic-event probability-t tizedére és figyeld az 500 000 futás eseményszámát.
2. Növeld N-t és figyeld a standard errort.
3. Változtasd a Beta-eloszlások koncentrációját azonos átlag mellett.
4. Módosítsd a splitting szinteket és hasonlítsd a stabilitást.


## Összefoglalás

Monte Carlo általános, de ritka eseménynél drága. A következő tanegységben azt vizsgáljuk, hogyan ronthatják le a redundancia előnyeit a **függőségek és common-cause failure** események.
